# Recorte da cidade-alvo e exportação inicial

Este notebook prepara o primeiro recorte do POC a partir do ZIP original do snapshot mensal do CNPJ.

## Fluxo recomendado no Colab

1. Manter no Google Drive apenas o ZIP mensal original do snapshot.
2. Montar o Drive no Colab e copiar esse ZIP para `/content`.
3. Executar este notebook.
4. O notebook move o ZIP para `original`, extrai o conteúdo para `extraido` e remove a camada temporária de ZIPs internos.
5. Em seguida, lê os arquivos brutos e produz o primeiro recorte em `processado/recorte`.

Configuração atual do POC:

- cidade-alvo: `Praia Grande - SP`
- código do município no layout do CNPJ: `6921`

In [ ]:
from pathlib import Path
import shutil
import zipfile
from google.colab import drive

try:
    import polars as pl
except ImportError:
    !pip -q install polars pyarrow
    import polars as pl

SNAPSHOT_MES = '2026-04'
NOME_ARQUIVO_ZIP = '2026-04.zip'
DRIVE_RAIZ = Path('/content/drive/MyDrive/geoBusiness/dados_brutos/cnpj')
CAMINHO_ZIP_DRIVE = DRIVE_RAIZ / SNAPSHOT_MES / 'original' / NOME_ARQUIVO_ZIP
ENTRADA_LOCAL = Path('/content/entrada_cnpj')
CAMINHO_ZIP_PRINCIPAL = ENTRADA_LOCAL / NOME_ARQUIVO_ZIP
RAIZ_DADOS = Path('/content/dados_brutos/cnpj')
CODIGO_MUNICIPIO = '6921'

drive.mount('/content/drive')
ENTRADA_LOCAL.mkdir(parents=True, exist_ok=True)
if not CAMINHO_ZIP_DRIVE.exists():
    raise FileNotFoundError(f'Arquivo não encontrado no Drive: {CAMINHO_ZIP_DRIVE}')
if not CAMINHO_ZIP_PRINCIPAL.exists():
    shutil.copy2(CAMINHO_ZIP_DRIVE, CAMINHO_ZIP_PRINCIPAL)
CAMINHO_ZIP_PRINCIPAL

In [ ]:
FAMILIAS = {
    'Empresas': 'empresas',
    'Estabelecimentos': 'estabelecimentos',
    'Socios': 'socios',
    'Simples': 'simples',
    'Cnaes': 'dominios',
    'Motivos': 'dominios',
    'Municipios': 'dominios',
    'Naturezas': 'dominios',
    'Paises': 'dominios',
    'Qualificacoes': 'dominios',
}

def classificar_zip(nome: str) -> str:
    for prefixo, familia in FAMILIAS.items():
        if nome.startswith(prefixo):
            return familia
    raise ValueError(f'ZIP interno não reconhecido: {nome}')

def preparar_snapshot(caminho_zip_principal: Path, raiz_dados: Path, snapshot_mes: str) -> Path:
    if not caminho_zip_principal.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {caminho_zip_principal}')

    snapshot = raiz_dados / snapshot_mes
    pacote_original = snapshot / 'original'
    extraido_texto = snapshot / 'extraido'
    processado = snapshot / 'processado'
    metadados = snapshot / 'metadados'
    temporario = snapshot / '_tmp_subarquivos_zip'

    for pasta in [pacote_original, extraido_texto, processado / 'recorte', processado / 'preparado', processado / 'analitico', metadados, temporario]:
        pasta.mkdir(parents=True, exist_ok=True)

    destino_zip = pacote_original / caminho_zip_principal.name
    if caminho_zip_principal.resolve() != destino_zip.resolve():
        shutil.move(str(caminho_zip_principal), str(destino_zip))
    caminho_zip_principal = destino_zip

    if not any(extraido_texto.rglob('*')):
        with zipfile.ZipFile(caminho_zip_principal) as zip_principal:
            zip_principal.extractall(temporario)

        for zip_interno in sorted(temporario.glob('*.zip')):
            familia = classificar_zip(zip_interno.name)
            destino_familia = extraido_texto / familia
            destino_familia.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_interno) as zip_familia:
                zip_familia.extractall(destino_familia)

        shutil.rmtree(temporario)
    elif temporario.exists():
        shutil.rmtree(temporario)

    return snapshot

SNAPSHOT = preparar_snapshot(CAMINHO_ZIP_PRINCIPAL, RAIZ_DADOS, SNAPSHOT_MES)
SNAPSHOT

In [ ]:
colunas_empresas = [
    'cnpj_basico', 'razao_social', 'natureza_juridica', 'qualificacao_responsavel',
    'capital_social', 'porte_empresa', 'ente_federativo_responsavel'
]
colunas_estabelecimentos = [
    'cnpj_basico', 'cnpj_ordem', 'cnpj_dv', 'identificador_matriz_filial', 'nome_fantasia',
    'situacao_cadastral', 'data_situacao_cadastral', 'motivo_situacao_cadastral',
    'nome_cidade_exterior', 'pais', 'data_inicio_atividade', 'cnae_fiscal_principal',
    'cnaes_secundarios', 'tipo_logradouro', 'logradouro', 'numero', 'complemento', 'bairro',
    'cep', 'uf', 'municipio', 'ddd_1', 'telefone_1', 'ddd_2', 'telefone_2', 'ddd_fax', 'fax',
    'correio_eletronico', 'situacao_especial', 'data_situacao_especial'
]
colunas_simples = [
    'cnpj_basico', 'opcao_pelo_simples', 'data_opcao_pelo_simples', 'data_exclusao_do_simples',
    'opcao_pelo_mei', 'data_opcao_pelo_mei', 'data_exclusao_do_mei'
]

In [ ]:
def ler_familia(pasta: Path, colunas: list[str]) -> pl.DataFrame:
    arquivos = sorted([arquivo for arquivo in pasta.iterdir() if arquivo.is_file()])
    consultas = []
    for arquivo in arquivos:
        consultas.append(
            pl.scan_csv(
                arquivo,
                separator=';',
                has_header=False,
                quote_char='"',
                new_columns=colunas,
                infer_schema_length=0,
                encoding='latin1',
            )
        )
    return pl.concat(consultas).collect(streaming=True)

empresas = ler_familia(SNAPSHOT / 'extraido' / 'empresas', colunas_empresas)
estabelecimentos = ler_familia(SNAPSHOT / 'extraido' / 'estabelecimentos', colunas_estabelecimentos)
simples = ler_familia(SNAPSHOT / 'extraido' / 'simples', colunas_simples)

empresas.shape, estabelecimentos.shape, simples.shape

In [ ]:
recorte_estabelecimentos = estabelecimentos.filter(pl.col('municipio') == CODIGO_MUNICIPIO)
recorte_empresas = empresas.join(recorte_estabelecimentos.select('cnpj_basico').unique(), on='cnpj_basico', how='inner')
recorte_simples = simples.join(recorte_estabelecimentos.select('cnpj_basico').unique(), on='cnpj_basico', how='inner')

recorte_estabelecimentos.shape, recorte_empresas.shape, recorte_simples.shape

In [ ]:
saida_recorte = SNAPSHOT / 'processado' / 'recorte'
saida_recorte.mkdir(parents=True, exist_ok=True)

recorte_estabelecimentos.write_parquet(saida_recorte / 'estabelecimentos_municipio.parquet')
recorte_empresas.write_parquet(saida_recorte / 'empresas_municipio.parquet')
recorte_simples.write_parquet(saida_recorte / 'simples_municipio.parquet')

print('Artefatos recorte gerados com sucesso.')